In [86]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Sequential

In [87]:
# 1. Corpus (Finance Domain Text)
text = """
i like personal finance
i like corporate finance
i like investment banking
i like stock market
i like portfolio management
i like risk management
i like financial modeling
i like quantitative finance
i like algorithmic trading
i like accounting
i love personal finance
i love corporate finance
i love investment banking
i love stock market
i love portfolio management
i love risk management
i love financial modeling
i love quantitative finance
i love algorithmic trading
i love accounting
i am learning personal finance
i am learning corporate finance
i am learning investment banking
i am learning stock market
i am learning portfolio management
i am learning risk management
i am learning financial modeling
i am learning quantitative finance
i am learning algorithmic trading
i am learning accounting
personal finance is useful
corporate finance is interesting
investment banking is challenging
stock market is exciting
portfolio management is valuable
risk management is crucial
financial modeling is useful
quantitative finance is powerful
algorithmic trading is fast
accounting is essential
personal finance is easy to learn
financial modeling is easy to learn
"""

In [88]:
# 2. Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

print("Word Index:", tokenizer.word_index)
print("Vocabulary size:", total_words)

Word Index: {'i': 1, 'finance': 2, 'is': 3, 'like': 4, 'love': 5, 'am': 6, 'learning': 7, 'management': 8, 'personal': 9, 'financial': 10, 'modeling': 11, 'corporate': 12, 'investment': 13, 'banking': 14, 'stock': 15, 'market': 16, 'portfolio': 17, 'risk': 18, 'quantitative': 19, 'algorithmic': 20, 'trading': 21, 'accounting': 22, 'useful': 23, 'easy': 24, 'to': 25, 'learn': 26, 'interesting': 27, 'challenging': 28, 'exciting': 29, 'valuable': 30, 'crucial': 31, 'powerful': 32, 'fast': 33, 'essential': 34}
Vocabulary size: 35


In [89]:
# 3. SET WINDOW SIZE HERE
WINDOW_SIZE = 5  # Increase to 6, 7, 8, etc. as needed

sequences = []
for line in text.strip().split("\n"):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        start_idx = max(0, (i + 1) - (WINDOW_SIZE + 1))
        n_gram_sequence = tokens[start_idx : i + 1]
        sequences.append(n_gram_sequence)

max_len = WINDOW_SIZE + 1
sequences = np.array(pad_sequences(sequences, maxlen=max_len, padding="pre"))

X = sequences[:, :-1]
y = sequences[:, -1]

print(f"X shape: {X.shape}  (Samples: {X.shape[0]}, Window Size: {X.shape[1]})")
print(f"y shape: {y.shape}")

X shape: (136, 5)  (Samples: 136, Window Size: 5)
y shape: (136,)


In [90]:
# Split features (X) and targets (y)
X = sequences[:, :-1]
y = sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (136, 5)
y shape: (136,)


In [91]:
model = Sequential([
    Input(shape=(WINDOW_SIZE,)),  # Matches your window size
    Embedding(total_words, 32),
    SimpleRNN(64),
    Dense(total_words, activation="softmax")
])

In [92]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 5, 32)          │         1,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 35)             │         2,275 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,603 (37.51 KB)

 Trainable params: 9,603 (37.51 KB)

 Non-trainable params: 0 (0.00 B)

In [93]:
# 5. Train Model
history = model.fit(X, y, epochs=300, verbose=0)
print("\nTraining Completed\n")


Training Completed



In [94]:
def predict_next_word(seed_text):
    sequence = tokenizer.texts_to_sequences([seed_text])[0]
    sequence = pad_sequences([sequence], maxlen=WINDOW_SIZE, padding="pre")
    prediction = model.predict(sequence, verbose=0)
    word_id = np.argmax(prediction)
    for word, index in tokenizer.word_index.items():
        if index == word_id:
            return word
    return ""

In [95]:
# 7. Predictions & Evaluations
print("--- Single Word Predictions ---")
print("i like ->", predict_next_word("i like"))
print("i love ->", predict_next_word("i love"))
print("i am ->", predict_next_word("i am"))
print("personal finance ->", predict_next_word("personal finance"))
print("investment ->", predict_next_word("investment"))

print("\n--- Complete Sentence Generation ---")
print("i like ->", generate_text("i like", 15))
print("personal finance ->", generate_text("personal finance", 10))

--- Single Word Predictions ---
i like -> stock
i love -> stock
i am -> learning
personal finance -> is
investment -> banking

--- Complete Sentence Generation ---
i like -> i like stock market is exciting to learn is valuable algorithmic trading is fast to learn exciting
personal finance -> personal finance is easy to learn is valuable to management is crucial


## **Predict previous word**

In [96]:
# Create sequences where X = following words and y = target previous word
X_prev_list = []
y_prev_list = []

for line in text.strip().split("\n"):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(len(tokens) - 1):
        y_target = tokens[i]  # Word to predict (preceding word)
        x_context = tokens[i + 1 : i + 1 + WINDOW_SIZE]  # Following context words
        X_prev_list.append(x_context)
        y_prev_list.append(y_target)

X_prev = pad_sequences(X_prev_list, maxlen=WINDOW_SIZE, padding="pre")
y_prev = np.array(y_prev_list)

print("X_prev shape:", X_prev.shape)
print("y_prev shape:", y_prev.shape)

X_prev shape: (136, 5)
y_prev shape: (136,)


In [97]:
# Build separate RNN model for backward prediction
model_prev = Sequential([
    Input(shape=(WINDOW_SIZE,)),
    Embedding(total_words, 32),
    SimpleRNN(64),
    Dense(total_words, activation="softmax")
])

model_prev.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_prev.fit(X_prev, y_prev, epochs=300, verbose=0)
print("Previous Word Model Training Completed!")

Previous Word Model Training Completed!


In [98]:
def predict_previous_word(following_text):
    sequence = tokenizer.texts_to_sequences([following_text])[0]
    sequence = pad_sequences([sequence], maxlen=WINDOW_SIZE, padding="pre")
    prediction = model_prev.predict(sequence, verbose=0)
    word_id = np.argmax(prediction)
    for word, index in tokenizer.word_index.items():
        if index == word_id:
            return word
    return ""

def generate_backward_text(seed_text, n_words):
    result = seed_text
    for _ in range(n_words):
        prev_word = predict_previous_word(result)
        if not prev_word:
            break
        result = prev_word + " " + result  # Prepend the predicted word
    return result

In [99]:
print("--- Previous Word Predictions ---")
print("<- personal finance :", predict_previous_word("personal finance"))
print("<- stock market :", predict_previous_word("stock market"))
print("<- banking is challenging :", predict_previous_word("banking is challenging"))

print("\n--- Backward Sentence Generation ---")
print("<- personal finance :", generate_backward_text("personal finance", 2))
print("<- easy to learn :", generate_backward_text("easy to learn", 3))

--- Previous Word Predictions ---
<- personal finance : learning
<- stock market : like
<- banking is challenging : investment

--- Backward Sentence Generation ---
<- personal finance : am learning personal finance
<- easy to learn : personal finance is easy to learn
